# E5 pilot timing on Kaggle T4
Thin notebook: all logic lives in `src/shattered`. Settings: GPU T4 x2 (one is used), Internet on.
Set `REPO_SRC` to a git URL, or attach the repo as a Kaggle dataset and point it at that path.

In [ ]:
REPO_SRC = 'https://github.com/<user>/shattered-scripts.git'  # or '/kaggle/input/shattered-scripts'
import os, subprocess
if REPO_SRC.startswith('http'):
    subprocess.run(['git', 'clone', REPO_SRC, '/kaggle/working/shattered-scripts'], check=True)
else:
    subprocess.run(['cp', '-r', REPO_SRC, '/kaggle/working/shattered-scripts'], check=True)
os.chdir('/kaggle/working/shattered-scripts')

In [ ]:
# keep Kaggle's torch; install the rest
!pip install -q -r requirements.in && pip install -q -e .

In [ ]:
# HF token from Kaggle secrets (Add-ons > Secrets > HF_TOKEN)
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

In [ ]:
!python scripts/env_check.py

In [ ]:
# download TinyStories V2 (pinned revision) and build tokenizers + streams (~20-40 min on Kaggle CPU)
!python -c "from huggingface_hub import hf_hub_download as d; [d('roneneldan/TinyStories', f, repo_type='dataset', revision='f54c09fd23315a6f9c86f9dc80f725de7d8f9c64', local_dir='data/raw/tinystories') for f in ['TinyStoriesV2-GPT4-valid.txt','TinyStoriesV2-GPT4-train.txt']]"
!python scripts/e05_pretrain.py prepare

In [ ]:
# T4 has no bf16: the training loop falls back to fp16 + GradScaler automatically
!python scripts/e05_pretrain.py pilot

In [ ]:
!cat results/e05_pretrain/pilot_report.json | head -80